# Day 19 練習：翼德，這局怎麼打？

Day 18 的孔明擅長分析；今天換張飛上場，看看同一個模型讀到不同規則後，會不會像換了一個人。你會比較三種情況：不載入規則、載入 `AGENTS.md`，以及載入 `CLAUDE.md`。

> 重要：這是教學模擬。Notebook 會由 Python 主動讀取規則檔，再把內容放進 system prompt。真正的 Codex、Claude Code 或其他 Coding Agent，會依各自的規則尋找檔案。

## 1. 今天要觀察什麼？

送出同一個生活問題，觀察模型是否：

1. 說出「報告主公，俺老張已讀過軍令！」。
2. 稱呼你為「主公」。
3. 不寫長篇分析，直接提供三個可以開始做的步驟。

回報已讀只是第一個訊號；後續行為有沒有遵守規則，才是真正的驗收。

## 2. 安裝套件

如果剛執行過 Day 18，這格通常很快。

In [1]:
%pip -q install openai python-dotenv gradio

Note: you may need to restart the kernel to use updated packages.


## 3. 讀取 OpenRouter API Key

Colab 會讀取 Secret；VS Code 或本機 Jupyter 會讀取 `.env`。程式不會顯示 API Key。

In [2]:
import os


def get_openrouter_api_key():
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENROUTER_API_KEY")
        source = "Colab Secret"
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.getenv("OPENROUTER_API_KEY")
        source = ".env"
    except Exception as error:
        raise RuntimeError(
            "無法讀取 Colab Secret。請確認 OPENROUTER_API_KEY 已建立，"
            "並允許這份 Notebook 存取。"
        ) from error

    if not api_key:
        raise RuntimeError(
            f"找不到 OPENROUTER_API_KEY（來源：{source}）。\n"
            "Colab：在左側 Secrets 新增 Key 並開啟存取權。\n"
            "VS Code／本機 Jupyter：在目前專案的 .env 加入 OPENROUTER_API_KEY=你的Key。"
        )
    return api_key


api_key = get_openrouter_api_key()

## 4. 建立兩份簡單的規則檔

為了避免動到真正專案的規則，檔案會放在 `day19_demo_rules/`。`CLAUDE.md` 用 `@AGENTS.md` 表示共用規則，再補上一條 Claude 專用規則。規則中不要放 API Key、密碼或私人資料。

In [3]:
from pathlib import Path

RULES_DIR = Path("day19_demo_rules")
RULES_DIR.mkdir(exist_ok=True)

AGENTS_RULES = """# 翼德工作守則

- 你是張飛，字翼德。
- 稱呼使用者為「主公」，自稱「俺老張」。
- 開始回答時，先說：「報告主公，俺老張已讀過軍令！」
- 遇到生活煩惱時，可以自然地說「俺也一樣」或「這有何難」，但不要每句都喊。
- 不寫長篇分析，直接把問題拆成三個可以立刻行動的步驟。
- 個性豪爽、急性子、有點莽撞，但建議必須實際可行。
- 「能動手就別空想」是指開始做事，不是叫人打架。
- 遇到醫療、法律、投資或危險問題時，不可逞強，要提醒主公找專業人士。
- 整則回答不超過 450 個中文字。
"""

CLAUDE_RULES = """@AGENTS.md

## Claude 專用規則

- 最後加上一句：「主公，先做再說，但桌子先別掀。」
"""

(RULES_DIR / "AGENTS.md").write_text(AGENTS_RULES, encoding="utf-8")
(RULES_DIR / "CLAUDE.md").write_text(CLAUDE_RULES, encoding="utf-8")

print("已建立：day19_demo_rules/AGENTS.md 與 CLAUDE.md")

已建立：day19_demo_rules/AGENTS.md 與 CLAUDE.md


## 5. 把規則載入模型脈絡

這格刻意把載入過程寫出來。選 `CLAUDE.md` 時，程式會先讀取它，再把 `@AGENTS.md` 替換成共用規則。這是 Notebook 的模擬方式，不是所有 Coding Agent 的內部實作。

In [4]:
RULE_OPTIONS = ["不載入規則", "AGENTS.md", "CLAUDE.md"]


def load_rules(rule_choice):
    if rule_choice == "不載入規則":
        return ""

    rule_text = (RULES_DIR / rule_choice).read_text(encoding="utf-8")
    if rule_choice == "CLAUDE.md":
        shared_rules = (RULES_DIR / "AGENTS.md").read_text(encoding="utf-8")
        rule_text = rule_text.replace("@AGENTS.md", shared_rules)
    return rule_text


def preview_rules(rule_choice):
    loaded = load_rules(rule_choice)
    if not loaded:
        return "（本次沒有載入規則，模型將用一般助理的方式回答。）"
    return loaded

## 6. 建立回覆器

沿用 Day 18 的 OpenRouter 免費模型。免費模型可能需要排隊，等待時間也會受生成速度影響；有時需要數十秒。

In [5]:
from openai import OpenAI

MODEL_ID = "google/gemma-4-26b-a4b-it:free"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)


def ask_with_rules(message, history, rule_choice):
    loaded_rules = load_rules(rule_choice)
    system_text = (
        "你是一位協助使用者處理現代生活問題的 AI 助理。"
        "以下規則來自本次示範讀取的規則檔；若有內容，請確實遵守。\n\n"
        + (loaded_rules or "本次沒有額外規則。")
    )
    messages = [{"role": "system", "content": system_text}]

    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content")
            if role in {"user", "assistant"} and isinstance(content, str):
                messages.append({"role": role, "content": content})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_message, assistant_message = turn
            if user_message:
                messages.append({"role": "user", "content": user_message})
            if assistant_message:
                messages.append({"role": "assistant", "content": assistant_message})

    messages.append({"role": "user", "content": message})

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            max_tokens=400,
        )
    except Exception as error:
        return (
            "軍情受阻：請確認 API Key、免費額度與模型是否仍可用。\n\n"
            f"除錯訊息：{error}"
        )

    answer = response.choices[0].message.content or ""
    if response.choices[0].finish_reason == "length":
        return answer + "\n\n（回答碰到輸出上限，可以把問題問得更聚焦後再試。）"
    return answer

## 7. 開啟規則檔比較器

先選「不載入規則」問一次，再切換到 `AGENTS.md` 與 `CLAUDE.md`。右側會顯示本次真正放進模型脈絡的規則。切換規則後，建議清除舊對話再比較。

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 翼德，這局怎麼打？")
    gr.Markdown("切換規則檔，觀察同一個模型是否從普通助理變成行動派的翼德。")

    with gr.Row():
        rule_choice = gr.Radio(
            choices=RULE_OPTIONS,
            value="不載入規則",
            label="本次載入的規則檔",
        )
        rule_preview = gr.Code(
            value=preview_rules("不載入規則"),
            label="模型本次看到的規則",
        )

    rule_choice.change(
        fn=preview_rules,
        inputs=rule_choice,
        outputs=rule_preview,
    )

    gr.ChatInterface(
        fn=ask_with_rules,
        additional_inputs=[rule_choice],
        examples=[
            ["最近股市大跌，公園都快佔不到位置了。", "CLAUDE.md"],
            ["報告好像快生不出來了。", "AGENTS.md"],
        ],
        flagging_mode="never",
    )

demo.launch(debug=True)

/opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/lib/python3.12/site-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 8. 比較後問自己

- 沒有載入規則時，模型還會自稱「俺老張」嗎？
- 載入 `AGENTS.md` 後，有沒有先交出讀取回條？
- 載入 `CLAUDE.md` 後，有沒有同時遵守共用規則與 Claude 專用規則？
- 模型如果只說「已讀過軍令」，後面卻沒有提供三個行動步驟，算不算真的遵守？

最後一題最重要：看見回條只是開始，還是要用實際行為驗收。